# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
!git clone https://github.com/ainasarfaraz343-a11y/flyrank-internship.git
%cd flyrank-internship

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 133 (delta 42), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.87 MiB | 10.39 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/flyrank-internship/flyrank-internship/flyrank-internship/flyrank-internship


In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Using **Logistic Regression** to predict `is_declining` (from `trend_direction
== 'down'`). The target is binary, coefficients stay interpretable (I can name
which signals push toward "declining"), and it's the honest starting point per
the skill's guidance: readable model first, add complexity only if the
comparison earns it. Random seed fixed at 42 throughout for reproducibility.

In [25]:
print(df['is_declining'].value_counts(normalize=True).round(3))

is_declining
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped split by `client_id` (GroupShuffleSplit, 80/20, seed=42) — not a random
row split. Rows from the same client share business patterns; letting one
client appear in both train and test would leak client identity into the score.
`avg_position == 0` rows (no GSC data, 1,205 rows) are dropped rather than
treated as position zero. `trend_direction`/`trend_pct` are excluded from
features — they define the label.

In [26]:
feature_cols = ['days_since_last_update', 'impressions_90d', 'clicks_90d',
                 'ctr', 'avg_position', 'engagement_rate', 'word_count',
                 'search_volume', 'content_age_days']

model_df = df[df['avg_position'] > 0].copy()

for col in ['word_count', 'search_volume', 'engagement_rate']:
    model_df[f'has_{col}'] = model_df[col].notna().astype(int)
    model_df[col] = model_df[col].fillna(0)

feature_cols = feature_cols + ['has_word_count', 'has_search_volume', 'has_engagement_rate']

X = model_df[feature_cols]
y = model_df['is_declining']
groups = model_df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(model_df.iloc[train_idx]['client_id']) & set(model_df.iloc[test_idx]['client_id'])
print(f"Train: {len(X_train)}, Test: {len(X_test)}, Client overlap: {overlap}")


Train: 22974, Test: 5821, Client overlap: set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same test rows, same metric: **precision@50**, plus base rate for context. The
Week-4 rule (`is_stale × has_volume`) is re-applied to this exact test split
so the comparison is apples-to-apples — same data, same split, same metric.

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_scaled, y_train)
model_scores = clf.predict_proba(X_test_scaled)[:, 1]

test_full = model_df.iloc[test_idx].copy()
test_full['is_stale'] = (test_full['days_since_last_update'] >= 90).astype(int)
test_full['has_volume'] = (test_full['impressions_90d'] >= 300).astype(int)
baseline_scores = test_full['is_stale'] * test_full['has_volume'] * test_full['impressions_90d']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k = 50
base_rate = y_test.mean()
model_p50 = precision_at_k(model_scores, y_test.values, k)
baseline_p50 = precision_at_k(baseline_scores.values, y_test.values, k)
model_auc = roc_auc_score(y_test, model_scores)

comparison = pd.DataFrame({
    'method': ['Week-4 rule baseline', 'Logistic Regression'],
    f'precision@{k}': [baseline_p50, model_p50],
    'ROC-AUC': [np.nan, round(model_auc, 3)],
})
print(f"Base rate (random guessing): {base_rate:.3f}")
print(comparison)

Base rate (random guessing): 0.540
                 method  precision@50  ROC-AUC
0  Week-4 rule baseline          0.28      NaN
1   Logistic Regression          0.44    0.559


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

**Top features:** The two strongest predictors are `content_age_days` (-0.263)
and `clicks_90d` (-0.220) — both pull a page away from being flagged as
declining. That makes sense: content that's been around a while has usually
already found its stable audience, and a page still getting clicks is clearly
still working. What stood out to me was `has_search_volume` (+0.192) and
`has_word_count` (+0.135) showing up near the top — pages where this keyword
data was originally missing turned out more likely to decline. That lines up
with what the data dictionary warned about: missing data isn't random here,
it's tied to content type, so treating "missing" as its own signal (instead of
just filling it with 0 and ignoring it) actually paid off. `has_engagement_rate`
sits at exactly 0.000 simply because that column had no missing values to
begin with. None of the coefficients are large enough to raise a leakage
concern — everything here is a modest, believable number.

**Where the model struggles:** Accuracy drops the most on fresh content
(freshness_tier 0-30, only 0.538 — barely better than a coin flip) and on
pages with "good" or "low" impressions (0.516, 0.526). It does noticeably
better on very stale pages (181+, 0.714). My read on this: for recently
updated, mid-traffic pages, decline is probably being driven by things this
model can't see at all — a competitor outranking them, a SERP layout change —
not staleness or volume.

**Three wrong cases:** All three share one thing — they were updated recently
(8-14 days ago), so on paper they look "fresh" and safe. One of them (row
3112) actually was declining, but the model gave it a fairly low risk score
(0.475) — it missed this one because a fresh page can still decline for
reasons outside what we're measuring. The other two were false alarms: both
had 0.00 CTR, and it looks like the model leaned hard on that low-click signal
and predicted decline anyway, even though the pages were fresh. So the model
isn't wrong randomly — it's consistently getting tripped up by fresh pages
with unusual click behavior.

**Overall:** Logistic Regression beats the Week-4 rule at precision@50 (0.44
vs 0.28) on the corrected test set, but its ROC-AUC of 0.559 is only just
above chance — so it's helpful for prioritizing the very top of a list, but
I wouldn't trust it as a general classifier yet. Both scores are lower than
what I saw before fixing the missing-data handling, which makes sense: the
rows I was accidentally dropping earlier turned out to be genuinely harder
to predict, so this is the more honest number.

In [28]:
# 1. Feature importance (coefficients)
coef_table = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': clf.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)
print("Feature importance (by |coefficient|):")
print(coef_table)
# 2. Where is the model most wrong?
test_full['predicted_prob'] = model_scores
test_full['predicted_label'] = (model_scores >= 0.5).astype(int)
test_full['actual_label'] = y_test.values
test_full['correct'] = test_full['predicted_label'] == test_full['actual_label']

print("Accuracy by freshness_tier:")
print(test_full.groupby('freshness_tier')['correct'].mean().round(3))
print()
print("Accuracy by impression_tier:")
print(test_full.groupby('impression_tier')['correct'].mean().round(3))
print()

# 3. Three concrete wrong cases
wrong_cases = test_full[~test_full['correct']].sample(3, random_state=42)
print("Three wrong cases:")
print(wrong_cases[['days_since_last_update', 'impressions_90d', 'ctr',
                     'avg_position', 'predicted_prob', 'actual_label']])


Feature importance (by |coefficient|):
                   feature  coefficient
8         content_age_days    -0.263444
2               clicks_90d    -0.219968
10       has_search_volume     0.192482
4             avg_position    -0.190512
3                      ctr    -0.175379
0   days_since_last_update     0.137014
9           has_word_count     0.134905
6               word_count     0.040155
5          engagement_rate    -0.039318
1          impressions_90d    -0.006105
7            search_volume     0.004963
11     has_engagement_rate     0.000000
Accuracy by freshness_tier:
freshness_tier
0-30      0.538
181+      0.714
31-90     0.700
91-180    0.550
Name: correct, dtype: float64

Accuracy by impression_tier:
impression_tier
excellent    0.612
good         0.516
low          0.526
moderate     0.573
Name: correct, dtype: float64

Three wrong cases:
       days_since_last_update  impressions_90d   ctr  avg_position  \
3112                       13              141  0.00          

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.